In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import os
from glaciation_time_estimator.auxiliary_func.config_reader import read_config

In [3]:
config = read_config("/cluster/work/climate/dnikolo/n2o/Glaciation_time_estimator/configs/Validation_DARDAR/2007_tracking/01_01.yaml")

In [5]:

ANALYSIS_YEAR=config["start_time"].year
OUTPUT_FOLDER=config["postprocessing_output_dir"]


In [3]:
ALLOWED_CTH_DEV=1000
def _check_val_seviri_coincidence(row):
    points_of_interest = row["val_cth_hist"] > 0
    if type(row["cth_hist"]) == float:
            N+=1
            return np.nan 
    cth_deviations = np.abs(row["cth_hist"][points_of_interest] - row["val_cth_hist"][points_of_interest])
    if cth_deviations.size == 0:
        return False
    mean_height_dev = cth_deviations.mean()  
    if mean_height_dev > ALLOWED_CTH_DEV:
        return False
    return True

def _get_val_seviri_coincidence(row):
    if "val_seviri_coincidence" not in row.keys():
        return _check_val_seviri_coincidence(row)
    else:
        return row["val_seviri_coincidence"]

def _identify_times_to_process(row):
    val_if_hist = row.val_ice_frac_hist
    idx = np.where((val_if_hist >= -0.01) & (val_if_hist <= 1.01))[0]
    return row.track_start_time+idx*pd.Timedelta(minutes=15)

cloud_data = pd.read_parquet("/nfs/ch4/wolke_scratch/dnikolo/GTE_Validation/DARDAR/Test_bulk/Results/Agg_03_glaciations_20070101.0000_20070115.0000.parquet")
cloud_data["val_seviri_coincidence"] = cloud_data.apply(_get_val_seviri_coincidence,axis=1)
valid_clouds = cloud_data[cloud_data["val_seviri_coincidence"]==True]
valid_clouds["colocation_time"]=valid_clouds.apply(_identify_times_to_process, axis=1)

row_temp = valid_clouds.iloc[1]
# for col_time in row["colocation_time"]:
col_time = row_temp["colocation_time"][0]
dt = pd.Timestamp(col_time)
time_str = dt.strftime("%Y%m%d_%H%M")

path = f"/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_{time_str}.nc"
print(path)
ds = xr.open_dataset(path)

lat_bins = ds.lat_bin.values
lon_bins = ds.lon_bin.values

lat0 = lat_bins[0]
lon0 = lon_bins[0]

dlat = lat_bins[1] - lat_bins[0]
dlon = lon_bins[1] - lon_bins[0]

nlat = len(lat_bins)
nlon = len(lon_bins)





/scratch/tmp.2873696.dnikolo/ipykernel_4002162/3487645135.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_clouds["colocation_time"]=valid_clouds.apply(_identify_times_to_process, axis=1)


/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070106_1000.nc


In [4]:

def get_colocation_ind(row):
    lat_idx_list = []
    lon_idx_list = []
    for lat, lon in zip(row.val_intersec_lat,row.val_intersec_lon):
        lat_idx = np.rint((lat - lat0) / dlat).astype(np.int64)
        lon_idx = np.rint((lon - lon0) / dlon).astype(np.int64)

        valid = (
            (lat_idx >= 0) & (lat_idx < nlat) &
            (lon_idx >= 0) & (lon_idx < nlon)
        )
        lat_idx_list.append(lat_idx[valid])
        lon_idx_list.append(lon_idx[valid])
    return lat_idx_list,lon_idx_list
# col_time = row["colocation_time"][0]
def get_bulk_values(row):
    cb_iwc_means  = []
    ct_iwc_means = []
    for i in range(len(row["colocation_time"])):
        col_time = row["colocation_time"][i]
        dt = pd.Timestamp(col_time)
        time_str = dt.strftime("%Y%m%d_%H%M")

        path = f"/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_{time_str}.nc"
        print(path)
        ds = xr.open_dataset(path)
        cb_iwc_values = ds["cb_iwc_mean"].isel(
            lat_bin=xr.DataArray(row["lat_bin_idx"][i], dims="pixel"),
            lon_bin=xr.DataArray(row["lon_bin_idx"][i], dims="pixel"),
            )
        ct_iwc_values = ds["cb_top_iwc_mean"].isel(
            lat_bin=xr.DataArray(row["lat_bin_idx"][i], dims="pixel"),
            lon_bin=xr.DataArray(row["lon_bin_idx"][i], dims="pixel"),
        )
        cb_iwc_means.append(cb_iwc_values.mean(axis=2)[0])
        ct_iwc_means.append(ct_iwc_values.mean(axis=1)[0])
    return cb_iwc_means, ct_iwc_means


def extract_iwc_relative_to_peak(row):
    records = []

    track_start = pd.Timestamp(row["track_start_time"])
    peak_ind = int(row["Glac_peak_ind"])

    for i, iwc_profile in enumerate(row["iwc_bulk_means"]):
        col_time = pd.Timestamp(row["colocation_time"][i])

        # Index of the colocation time within the cloud track
        col_ind = int(round((col_time - track_start).total_seconds() / (15 * 60)))

        # Time index relative to the ice-fraction peak
        rel_ind = col_ind - peak_ind

        records.append({
            "Cloud_ID": row["Cloud_ID"],
            "rel_ind": rel_ind,
            "rel_time_min": rel_ind * 15,
            "iwc_profile": np.asarray(iwc_profile)
        })

    return records

In [5]:
valid_clouds[["lat_bin_idx", "lon_bin_idx"]] = (
    valid_clouds
    .apply(get_colocation_ind, axis=1, result_type="expand")
)

valid_clouds[["iwc_bulk_means","iwc_top_means"]] = (valid_clouds.apply(get_bulk_values, axis=1, result_type="expand"))


/scratch/tmp.2873696.dnikolo/ipykernel_4002162/3721039598.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_clouds[["lat_bin_idx", "lon_bin_idx"]] = (
/scratch/tmp.2873696.dnikolo/ipykernel_4002162/3721039598.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_clouds[["lat_bin_idx", "lon_bin_idx"]] = (


/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070106_1000.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070113_0315.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070103_0930.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070103_1415.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070104_0000.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070104_1645.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070106_1145.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070107_1215.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070107_2200.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070107_1400.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070107_2330.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070108_1300.nc
/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070108_2245.nc
/nfs/ch4/wolke_scratch/dn

/scratch/tmp.2873696.dnikolo/ipykernel_4002162/3721039598.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_clouds[["iwc_bulk_means","iwc_top_means"]] = (valid_clouds.apply(get_bulk_values, axis=1, result_type="expand"))
/scratch/tmp.2873696.dnikolo/ipykernel_4002162/3721039598.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_clouds[["iwc_bulk_means","iwc_top_means"]] = (valid_clouds.apply(get_bulk_values, axis=1, result_type="expand"))


In [6]:

all_records = []

for _, row in valid_clouds.iterrows():
    all_records.extend(extract_iwc_relative_to_peak(row))

iwc_rel_df = pd.DataFrame(all_records)
iwc_stats_by_rel_time = (
    iwc_rel_df
    .groupby("rel_ind")["iwc_profile"]
    .agg(
        n_hits="count",
        mean_iwc_profile=lambda profiles: np.nanmean(np.stack(profiles), axis=0),
    )
    .reset_index()
)
iwc_stats_by_rel_time["rel_time_min"] = iwc_stats_by_rel_time["rel_ind"] * 15



/scratch/tmp.2873696.dnikolo/ipykernel_4002162/3174747400.py:12: RuntimeWarning: Mean of empty slice
  mean_iwc_profile=lambda profiles: np.nanmean(np.stack(profiles), axis=0),


In [14]:
iwc_stats_by_rel_time

,rel_ind,n_hits,mean_iwc_profile,rel_time_min
0,-130,1,"[6.243813e-05, 7.337958e-05, 8.3111365e-05, 9....",-1950
1,-118,1,"[2.8283175e-06, 3.4134316e-06, 4.036847e-06, 4...",-1770
2,-109,1,"[1.1533018e-05, 1.1518599e-05, 1.1566741e-05, ...",-1635
3,-107,1,"[1.4763028e-05, 1.4167337e-05, 1.3642586e-05, ...",-1605
4,-86,1,"[8.7645935e-07, 8.57619e-07, 8.483577e-07, nan...",-1290
...,...,...,...,...
87,84,2,"[2.1365257e-05, 2.2563994e-05, 2.4541929e-05, ...",1260
88,85,2,"[1.4204555e-06, 1.4274059e-06, 1.5327861e-06, ...",1275
89,96,1,"[3.725859e-05, 4.0058258e-05, 4.4069202e-05, 4...",1440
90,137,1,"[3.5470778e-06, 3.973085e-06, 4.360458e-06, 4....",2055


In [19]:
df2 = iwc_stats_by_rel_time.set_index("rel_ind")



In [21]:
df2

,n_hits,mean_iwc_profile,rel_time_min
rel_ind,,,
-130,1,"[6.243813e-05, 7.337958e-05, 8.3111365e-05, 9....",-1950
-118,1,"[2.8283175e-06, 3.4134316e-06, 4.036847e-06, 4...",-1770
-109,1,"[1.1533018e-05, 1.1518599e-05, 1.1566741e-05, ...",-1635
-107,1,"[1.4763028e-05, 1.4167337e-05, 1.3642586e-05, ...",-1605
-86,1,"[8.7645935e-07, 8.57619e-07, 8.483577e-07, nan...",-1290
...,...,...,...
84,2,"[2.1365257e-05, 2.2563994e-05, 2.4541929e-05, ...",1260
85,2,"[1.4204555e-06, 1.4274059e-06, 1.5327861e-06, ...",1275
96,1,"[3.725859e-05, 4.0058258e-05, 4.4069202e-05, 4...",1440


In [23]:
df2.loc[range(-3, 4)]

,n_hits,mean_iwc_profile,rel_time_min
rel_ind,,,
-3,2,"[5.0834387e-05, 6.040129e-05, 6.3510815e-05, 2...",-45
-2,4,"[3.8377933e-05, 4.0987456e-05, 4.4154964e-05, ...",-30
-1,3,"[2.3369948e-05, 2.6768534e-05, 2.9551995e-05, ...",-15
0,4,"[1.9410603e-05, 2.2126482e-05, 2.539643e-05, 2...",0
1,1,"[2.8202053e-06, 3.1562572e-06, 3.647534e-06, 3...",15
2,1,"[1.1652407e-05, 1.1954662e-05, 1.2120627e-05, ...",30
3,1,"[1.2065032e-06, 1.2697072e-06, 9.762616e-07, 1...",45


In [17]:
iwc_stats_by_rel_time.loc[iwc_stats_by_rel_time["rel_ind"].isin(range(-5, 5))]

,rel_ind,n_hits,mean_iwc_profile,rel_time_min
41,-5,3,"[1.0902558e-05, 1.13755805e-05, 1.2206234e-05,...",-75
42,-4,2,"[2.123949e-05, 2.4962475e-05, 2.9050707e-05, 3...",-60
43,-3,2,"[5.0834387e-05, 6.040129e-05, 6.3510815e-05, 2...",-45
44,-2,4,"[3.8377933e-05, 4.0987456e-05, 4.4154964e-05, ...",-30
45,-1,3,"[2.3369948e-05, 2.6768534e-05, 2.9551995e-05, ...",-15
46,0,4,"[1.9410603e-05, 2.2126482e-05, 2.539643e-05, 2...",0
47,1,1,"[2.8202053e-06, 3.1562572e-06, 3.647534e-06, 3...",15
48,2,1,"[1.1652407e-05, 1.1954662e-05, 1.2120627e-05, ...",30
49,3,1,"[1.2065032e-06, 1.2697072e-06, 9.762616e-07, 1...",45


In [4]:
ds = xr.open_mfdataset("/nfs/ch4/wolke_scratch/dnikolo/dump/dardar-cloud_cb/DD_CB_20070106_1000.nc")

In [21]:
data = ds["cb_iwc_mean"].isel(
            lat_bin=xr.DataArray([1,2,3], dims="pixel"),
            lon_bin=xr.DataArray([1,2,3], dims="pixel")
        )
data.mean(axis=2)[0]

<xarray.DataArray 'cb_iwc_mean' (bulk_level: 5)> Size: 20B
dask.array<getitem, shape=(5,), dtype=float32, chunksize=(5,), chunktype=numpy.ndarray>
Coordinates:
    time_bin    datetime64[ns] 8B 2007-01-06T10:00:00
  * bulk_level  (bulk_level) int16 10B 0 1 2 3 4

In [23]:
data = ds["cb_top_iwc_mean"].isel(
            lat_bin=xr.DataArray([1,2,3], dims="pixel"),
            lon_bin=xr.DataArray([1,2,3], dims="pixel")
        )
data.mean(axis=1)[0]


<xarray.DataArray 'cb_top_iwc_mean' ()> Size: 4B
dask.array<getitem, shape=(), dtype=float32, chunksize=(), chunktype=numpy.ndarray>
Coordinates:
    time_bin  datetime64[ns] 8B 2007-01-06T10:00:00